# Grazioso Salvare Dashboard — CS 499 Enhanced Version

This notebook is an enhanced copy of the original CS 340 Project Two dashboard.

## Enhancements demonstrated

- Secure environment-based MongoDB configuration
- Immediate database connection verification
- DataFrame-based retrieval through `AnimalShelter_enhanced.py`
- Database-side rescue filtering, projection, sorting, and result limits
- Dictionary-based rescue-query selection
- Structured callback error handling
- Visible database and result-status messages
- Responsive map and chart layouts
- Safer coordinate validation
- Cleaner application initialization with modern Dash
- Improved code organization and maintainability

The original `ProjectTwoDashboard.ipynb` remains unchanged for before-and-after comparison.


## Local setup

Place these items in the same project directory:

```text
ProjectTwoDashboard_enhanced.ipynb
AnimalShelter_enhanced.py
assets/
    GraziosoSalvareLogo.png
```

Set the MongoDB environment variables before running the notebook:

```text
MONGO_USERNAME=aacuser
MONGO_PASSWORD=your_password
MONGO_HOST=localhost
MONGO_PORT=27017
MONGO_AUTH_SOURCE=admin
MONGO_DB_NAME=aac
MONGO_COLLECTION_NAME=animals
MONGO_TLS=false
MONGO_USE_SRV=false
MONGO_TIMEOUT_MS=5000
```

A local `.env` file may be used when `python-dotenv` is installed. Do not commit `.env` to GitHub.


In [ ]:
################################################################################
# Imports, constants, and logging
################################################################################

from pathlib import Path
import logging
from typing import Any, Dict

import pandas as pd
import plotly.express as px
import dash_leaflet as dl
from dash import Dash, Input, Output, dcc, html, dash_table
from pymongo import ASCENDING

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    # Environment variables can still be supplied through Windows or the shell.
    pass

from AnimalShelter_enhanced import (
    AnimalShelter,
    AnimalShelterError,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

LOGGER = logging.getLogger("grazioso_dashboard")
logging.getLogger("werkzeug").setLevel(logging.ERROR)

UNIQUE_ID = "ml-grazioso-dash-enhanced"
DASHBOARD_LIMIT = 500
DEFAULT_CENTER = [30.75, -97.48]

DISPLAY_COLUMNS = [
    "animal_id",
    "name",
    "animal_type",
    "breed",
    "sex_upon_outcome",
    "age_upon_outcome",
    "age_upon_outcome_in_weeks",
    "outcome_type",
    "location_lat",
    "location_long",
]

DISPLAY_PROJECTION = {column: 1 for column in DISPLAY_COLUMNS}
DISPLAY_PROJECTION["_id"] = 0


In [ ]:
################################################################################
# Rescue-category query definitions
################################################################################

RESCUE_QUERIES: Dict[str, Dict[str, Any]] = {
    "reset": {},
    "water": {
        "breed": {
            "$in": [
                "Labrador Retriever Mix",
                "Chesapeake Bay Retriever",
                "Newfoundland",
            ]
        },
        "sex_upon_outcome": {
            "$regex": "^Intact",
            "$options": "i",
        },
        "age_upon_outcome_in_weeks": {"$lte": 156},
    },
    "mountain": {
        "breed": {
            "$in": [
                "German Shepherd",
                "Alaskan Malamute",
                "Old English Sheepdog",
                "Siberian Husky",
                "Rottweiler",
            ]
        },
        "sex_upon_outcome": {
            "$regex": "^Intact",
            "$options": "i",
        },
        "age_upon_outcome_in_weeks": {"$lte": 156},
    },
    "disaster": {
        "breed": {
            "$in": [
                "Doberman Pinscher",
                "German Shepherd",
                "Golden Retriever",
                "Bloodhound",
                "Rottweiler",
            ]
        },
        "sex_upon_outcome": {
            "$regex": "^Intact",
            "$options": "i",
        },
        "age_upon_outcome_in_weeks": {"$lte": 156},
    },
}

RESCUE_LABELS = {
    "reset": "All available animals",
    "water": "Water rescue",
    "mountain": "Mountain or wilderness rescue",
    "disaster": "Disaster or individual tracking",
}


In [ ]:
################################################################################
# Secure MongoDB connection
################################################################################

try:
    shelter = AnimalShelter.from_env(
        verify_connection=True,
        audit_fields=True,
    )
    DATABASE_ONLINE = shelter.health_check()
    ACTIVE_RECORD_COUNT = shelter.count()
except AnimalShelterError as exc:
    shelter = None
    DATABASE_ONLINE = False
    ACTIVE_RECORD_COUNT = 0
    LOGGER.error("Dashboard database initialization failed: %s", exc)

connection_text = (
    f"Database connected | Active records: {ACTIVE_RECORD_COUNT:,}"
    if DATABASE_ONLINE
    else "Database unavailable — check MongoDB and environment variables"
)

connection_text


### Optional index setup

Run the next cell once after reviewing the existing data. The unique `animal_id` index remains disabled until duplicate values have been checked.


In [ ]:
# Optional: create common query indexes once.
# Leave commented until you are ready to modify the local database.
#
# if shelter is not None:
#     created_indexes = shelter.ensure_indexes(
#         unique_animal_id=False,
#         include_common_indexes=True,
#     )
#     print(created_indexes)


In [ ]:
################################################################################
# Data-access helper functions
################################################################################

def empty_display_dataframe() -> pd.DataFrame:
    """Return an empty DataFrame with stable dashboard columns."""
    return pd.DataFrame(columns=DISPLAY_COLUMNS)


def query_for_rescue_type(rescue_value: str) -> pd.DataFrame:
    """
    Retrieve one rescue category as a Pandas DataFrame.

    Filtering, projection, sorting, and limiting are performed by MongoDB before
    the results are converted to a DataFrame.
    """
    if shelter is None:
        return empty_display_dataframe()

    query = RESCUE_QUERIES.get(rescue_value, {})

    try:
        dataframe = shelter.read_dataframe(
            query=query,
            projection=DISPLAY_PROJECTION,
            limit=DASHBOARD_LIMIT,
            sort=[("breed", ASCENDING), ("name", ASCENDING)],
        )
    except AnimalShelterError as exc:
        LOGGER.error("Unable to retrieve rescue data: %s", exc)
        return empty_display_dataframe()

    for column in DISPLAY_COLUMNS:
        if column not in dataframe.columns:
            dataframe[column] = pd.NA

    return dataframe[DISPLAY_COLUMNS]


df_master = query_for_rescue_type("reset")
(len(df_master), list(df_master.columns))


In [ ]:
################################################################################
# Dash application and responsive layout
################################################################################

assets_path = Path("assets")
logo_candidates = [
    "GraziosoSalvareLogo.png",
    "Grazioso Salvare Logo.png",
]
logo_filename = next(
    (name for name in logo_candidates if (assets_path / name).exists()),
    logo_candidates[0],
)

app = Dash(
    __name__,
    assets_folder=str(assets_path),
    title="Grazioso Salvare Dashboard",
)

logo = html.Img(
    src=app.get_asset_url(logo_filename),
    alt="Grazioso Salvare logo",
    style={
        "maxHeight": "90px",
        "maxWidth": "100%",
        "margin": "8px 0",
    },
)

app.layout = html.Div(
    style={
        "fontFamily": "Arial, sans-serif",
        "padding": "16px",
        "maxWidth": "1500px",
        "margin": "0 auto",
    },
    children=[
        html.Header(
            [
                html.H1(
                    "Grazioso Salvare Dashboard — Enhanced",
                    style={"textAlign": "center"},
                ),
                html.Div(logo, style={"textAlign": "center"}),
                html.Div(
                    f"Unique ID: {UNIQUE_ID}",
                    style={
                        "textAlign": "center",
                        "fontWeight": "bold",
                        "marginBottom": "8px",
                    },
                ),
                html.Div(
                    connection_text,
                    id="database-status",
                    role="status",
                    style={
                        "textAlign": "center",
                        "fontWeight": "bold",
                        "padding": "8px",
                    },
                ),
            ]
        ),
        html.Hr(),
        html.H3("Select Rescue Type"),
        dcc.RadioItems(
            id="filter-type",
            options=[
                {"label": "Water Rescue", "value": "water"},
                {
                    "label": "Mountain or Wilderness Rescue",
                    "value": "mountain",
                },
                {
                    "label": "Disaster or Individual Tracking",
                    "value": "disaster",
                },
                {"label": "Reset (All)", "value": "reset"},
            ],
            value="reset",
            labelStyle={
                "display": "block",
                "margin": "6px 0",
            },
        ),
        html.Div(
            id="status-message",
            role="status",
            style={
                "fontWeight": "bold",
                "margin": "12px 0",
                "minHeight": "24px",
            },
        ),
        dash_table.DataTable(
            id="datatable-id",
            columns=[
                {
                    "name": column.replace("_", " ").title(),
                    "id": column,
                    "selectable": True,
                }
                for column in DISPLAY_COLUMNS
            ],
            data=df_master.to_dict("records"),
            row_selectable="single",
            selected_rows=[0] if not df_master.empty else [],
            selected_columns=[],
            filter_action="native",
            sort_action="native",
            sort_mode="multi",
            page_action="native",
            page_current=0,
            page_size=10,
            style_table={
                "overflowX": "auto",
                "border": "1px solid #ddd",
            },
            style_header={
                "fontWeight": "bold",
                "textAlign": "left",
            },
            style_cell={
                "minWidth": "120px",
                "maxWidth": "260px",
                "whiteSpace": "normal",
                "height": "auto",
                "padding": "6px",
                "textAlign": "left",
            },
        ),
        html.Br(),
        html.Div(
            style={
                "display": "grid",
                "gridTemplateColumns": "repeat(auto-fit, minmax(360px, 1fr))",
                "gap": "18px",
                "alignItems": "start",
            },
            children=[
                html.Div(id="map-id"),
                html.Div(dcc.Graph(id="second-chart-id")),
            ],
        ),
    ],
)

app


In [ ]:
################################################################################
# Table filtering and visible status
################################################################################

@app.callback(
    Output("datatable-id", "data"),
    Output("datatable-id", "selected_rows"),
    Output("status-message", "children"),
    Input("filter-type", "value"),
)
def update_table(filter_value):
    dataframe = query_for_rescue_type(filter_value)
    label = RESCUE_LABELS.get(filter_value, "Selected rescue category")

    if dataframe.empty:
        return [], [], f"{label}: no matching animals were found."

    limited_note = (
        f" The dashboard limit is {DASHBOARD_LIMIT:,} records."
        if len(dataframe) >= DASHBOARD_LIMIT
        else ""
    )
    message = f"{label}: {len(dataframe):,} animals displayed.{limited_note}"
    return dataframe.to_dict("records"), [0], message


In [ ]:
################################################################################
# Selected-column highlighting
################################################################################

@app.callback(
    Output("datatable-id", "style_data_conditional"),
    Input("datatable-id", "selected_columns"),
)
def highlight_selected_columns(selected_columns):
    selected_columns = selected_columns or []
    return [
        {
            "if": {"column_id": column},
            "backgroundColor": "#D2F3FF",
        }
        for column in selected_columns
    ]


In [ ]:
################################################################################
# Responsive animal-location map
################################################################################

def safe_float(value: Any, default: float) -> float:
    """Convert a value to float and reject missing or nonnumeric coordinates."""
    try:
        converted = float(value)
        if pd.isna(converted):
            return default
        return converted
    except (TypeError, ValueError):
        return default


def build_empty_map():
    """Build the default map displayed when no animal is selected."""
    return dl.Map(
        center=DEFAULT_CENTER,
        zoom=10,
        style={"width": "100%", "height": "450px"},
        children=[dl.TileLayer()],
    )


@app.callback(
    Output("map-id", "children"),
    Input("datatable-id", "derived_virtual_data"),
    Input("datatable-id", "derived_virtual_selected_rows"),
)
def update_map(view_data, selected_rows):
    dataframe = (
        pd.DataFrame(view_data)
        if view_data
        else empty_display_dataframe()
    )

    if dataframe.empty:
        return build_empty_map()

    row_index = selected_rows[0] if selected_rows else 0
    row_index = max(0, min(row_index, len(dataframe) - 1))
    animal = dataframe.iloc[row_index]

    latitude = safe_float(animal.get("location_lat"), DEFAULT_CENTER[0])
    longitude = safe_float(animal.get("location_long"), DEFAULT_CENTER[1])
    breed = str(animal.get("breed") or "Unknown breed")
    name = str(animal.get("name") or "Unnamed animal")
    animal_id = str(animal.get("animal_id") or "Unknown ID")

    return dl.Map(
        center=[latitude, longitude],
        zoom=10,
        style={"width": "100%", "height": "450px"},
        children=[
            dl.TileLayer(),
            dl.Marker(
                position=[latitude, longitude],
                children=[
                    dl.Tooltip(breed),
                    dl.Popup(
                        [
                            html.H4(name),
                            html.P(f"Animal ID: {animal_id}"),
                            html.P(f"Breed: {breed}"),
                        ]
                    ),
                ],
            ),
        ],
    )


In [ ]:
################################################################################
# Top-breeds chart
################################################################################

@app.callback(
    Output("second-chart-id", "figure"),
    Input("datatable-id", "derived_virtual_data"),
)
def update_breed_chart(view_data):
    dataframe = (
        pd.DataFrame(view_data)
        if view_data
        else empty_display_dataframe()
    )

    if dataframe.empty or "breed" not in dataframe.columns:
        figure = px.bar(title="No breed data available")
        figure.update_layout(
            xaxis_title="Breed",
            yaxis_title="Count",
        )
        return figure

    top_breeds = (
        dataframe["breed"]
        .fillna("Unknown")
        .astype(str)
        .value_counts()
        .head(15)
        .rename_axis("breed")
        .reset_index(name="count")
    )

    figure = px.bar(
        top_breeds,
        x="breed",
        y="count",
        title="Top 15 Breeds in the Current Selection",
    )
    figure.update_layout(
        xaxis_title="Breed",
        yaxis_title="Count",
        margin={"l": 20, "r": 20, "t": 55, "b": 110},
        xaxis_tickangle=-40,
    )
    return figure


## Run the dashboard

The following cell starts Dash inside Jupyter. Stop the cell before rerunning earlier setup cells.

For a browser tab instead of an inline display, change `jupyter_mode="inline"` to `jupyter_mode="external"`.


In [ ]:
app.run(
    jupyter_mode="inline",
    debug=False,
    port=8050,
)


## Cleanup

Run this cell when you are finished with the notebook so the MongoDB client closes cleanly.


In [ ]:
if shelter is not None:
    shelter.close()
    print("MongoDB connection closed.")
